**Dataset**
labeled datasset collected from Spotify (Assignment 1 - Spotify Reviews Rating)

**Objective**
classify Review to a category from 1 to 5. <br>

**Total Estimated Time = 90-120 Mins**

**Evaluation metric**
macro f1 score

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Import used libraries

In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import CountVectorizer
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn.pipeline import Pipeline
import nltk
from sklearn.preprocessing import FunctionTransformer
from nltk.corpus import stopwords
import seaborn as sns
import matplotlib.pyplot as plt
import re
from sklearn.metrics import f1_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import RandomizedSearchCV
from nltk.tokenize import word_tokenize
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_colwidth', 500)
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

### Load Dataset

In [ ]:
df=pd.read_csv('/content/drive/MyDrive/Spotify Reviews Rating.csv')
df.head()

,Time_submitted,Review,Rating
0,7/9/2022 15:00,"Great music service, the audio is high quality and the app is easy to use. Also very quick and friendly support.",5
1,7/9/2022 14:21,Please ignore previous negative rating. This app is super great. I give it five stars+,5
2,7/9/2022 13:27,"This pop-up ""Get the best Spotify experience on Android 12"" is too annoying. Please let's get rid of this.",4
3,7/9/2022 13:26,Really buggy and terrible to use as of recently,1
4,7/9/2022 13:20,Dear Spotify why do I get songs that I didn't put on my playlist??? And why do we have shuffle play?,1


### Data splitting

It is a good practice to split the data before EDA helps maintain the integrity of the machine learning process, prevents data leakage, simulates real-world scenarios more accurately, and ensures reliable model performance evaluation on unseen data.

In [ ]:
from os import X_OK
X=df['Review']
y=df['Rating']
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,stratify=y)

### EDA on training data

- check NaNs

In [ ]:
df.isna().sum()

,0
Time_submitted,0
Review,0
Rating,0


- check duplicates

In [ ]:
df.duplicated().sum()

np.int64(0)

- show a representative sample of data texts to find out required preprocessing steps

In [ ]:
df['Review'].head(5)

,Review
0,"Great music service, the audio is high quality and the app is easy to use. Also very quick and friendly support."
1,Please ignore previous negative rating. This app is super great. I give it five stars+
2,"This pop-up ""Get the best Spotify experience on Android 12"" is too annoying. Please let's get rid of this."
3,Really buggy and terrible to use as of recently
4,Dear Spotify why do I get songs that I didn't put on my playlist??? And why do we have shuffle play?


- check dataset balancing

In [ ]:
df['Rating'].value_counts()

,count
Rating,
5,22095
1,17653
4,7842
2,7118
3,6886


- Cleaning and Preprocessing are:
    - 1
    - 2
    - 3
    - ... etc.

### Cleaning and Preprocessing

In [ ]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def process_text(text):

  #clean text
  text = str(text).lower()
  text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
  text=re.sub(r"http\S+|www\S+|https\S+|@\w+","",text)

  #split words
  tokens=word_tokenize(text)

  #lemmetization
  tokens=[lemmatizer.lemmatize(word) for word in tokens
          if word not in stop_words]

  return ' '.join(tokens)

def text_cleaner(text_list):

    return pd.Series(text_list).apply(process_text)


preprocessor_step = FunctionTransformer(text_cleaner)


**You  are doing Great so far!**

### Modelling

In [ ]:
lg = LogisticRegression(multi_class='multinomial')

In [ ]:
model_pipeline = Pipeline(steps=[
    ("preprocessing", preprocessor_step),
    ("vectorizer", TfidfVectorizer()),
    ("classifier", lg)
])

In [ ]:
model_pipeline.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Pipeline(steps=[('preprocessing',
                 FunctionTransformer(func=<function text_cleaner at 0x7ea29453c900>)),
                ('vectorizer', TfidfVectorizer()),
                ('classifier', LogisticRegression(multi_class='multinomial'))])

#### Evaluation

**Evaluation metric:**
macro f1 score

Macro F1 score is a useful metric in scenarios where you want to evaluate the overall performance of a multi-class classification model, **particularly when the classes are imbalanced**

![Calculation](https://assets-global.website-files.com/5d7b77b063a9066d83e1209c/639c3d934e82c1195cdf3c60_macro-f1.webp)

In [ ]:
y_pred = model_pipeline.predict(X_test)

In [ ]:
f1_score(y_test, y_pred, average='macro')

0.4356060543317497

### Enhancement

In [ ]:
lgl = LogisticRegression(multi_class='multinomial',max_iter=1000,solver='lbfgs')

In [ ]:
model_pipeline = Pipeline(steps=[
    ("preprocessing", preprocessor_step),
    ("vectorizer", TfidfVectorizer(max_features=10000,ngram_range=(1,2))),
    ("classifier", lgl)
])

In [ ]:
model_pipeline.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Pipeline(steps=[('preprocessing',
                 FunctionTransformer(func=<function text_cleaner at 0x7ea29453c900>)),
                ('vectorizer',
                 TfidfVectorizer(max_features=10000, ngram_range=(1, 2))),
                ('classifier',
                 LogisticRegression(max_iter=5000, multi_class='multinomial'))])

In [ ]:
y_pred2 = model_pipeline.predict(X_test)

### Conclusion and final results


In [ ]:
f1_score(y_test, y_pred2, average='macro')

0.43517503620107884

#### Done!